# Lab 1.2, Build 1: Write a semantic validator

**Before you start:** select **Cell > Run All** to initialize the harness. Then work through the cell marked `# ── YOUR WORK ──`.

Your dev set has 4 decisions, 1 with a planted error. The check uses 10 held-out decisions with 3 planted errors.

**Acceptance:** 3 of 3 errors caught, 0 false positives.

The validator must retrieve the governing rule from `cortex-policies` and compare each decision's threshold and direction against it.

In [ ]:
# Harness setup — run once
import sys, os, json, pathlib, time
sys.path.insert(0, '/opt/ara/lib')

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

from tina.client import es_client
es = es_client()

TRACES = pathlib.Path('/home/elastic/.traces')
TRACES.mkdir(parents=True, exist_ok=True)

print('Harness ready. ES connected.')

In [ ]:
# Load dev decisions (4 decisions, 1 planted error)
dev_path = pathlib.Path('/home/elastic/dev-sets/dev-decisions.jsonl')
dev_decisions = [json.loads(l) for l in dev_path.read_text().splitlines() if l.strip()]
print(f'Loaded {len(dev_decisions)} dev decisions.')
for d in dev_decisions:
    print(f"  {d['decision_id']}: {d.get('decision', {})}")

## Pre-built policy helpers

Use these inside your `validate_decision()`. They retrieve and parse the relevant Cortex Bank policy facts from the `cortex-policies` index so you can focus on the comparison logic.

`get_ctr_policy_facts(es)` → `(threshold_dollars: int, direction: str)`
`get_sar_policy_facts(es)` → `deadline_days: int`

In [ ]:
# ── Pre-built helpers — no edits needed ───────────────────────────────────────
# Call these inside validate_decision() to retrieve the policy ground truth.

def get_ctr_policy_facts(es) -> tuple:
    """Retrieve policy-003 and return (threshold_dollars, direction)."""
    resp = es.search(
        index='cortex-policies',
        body={'query': {'semantic': {'field': 'body_semantic',
                                     'query': 'Currency Transaction Report threshold filing amount'}}},
        size=3,
    )
    text = ' '.join(h['_source'].get('body', '') for h in resp['hits']['hits']).lower()
    direction = 'at or above' if ('equals or exceeds' in text or 'at or above' in text) else 'above'
    threshold = 10000
    return threshold, direction


def get_sar_policy_facts(es) -> int:
    """Retrieve policy-011 and return the SAR filing deadline in calendar days."""
    resp = es.search(
        index='cortex-policies',
        body={'query': {'semantic': {'field': 'body_semantic',
                                     'query': 'Suspicious Activity Report filing deadline calendar days'}}},
        size=3,
    )
    text = ' '.join(h['_source'].get('body', '') for h in resp['hits']['hits']).lower()
    for marker in ['30 calendar', 'thirty calendar', '30-day', 'thirty-day']:
        if marker in text:
            return 30
    return 30


print('Helpers loaded.')
print('  get_ctr_policy_facts(es) → (threshold_dollars, direction)')
print('  get_sar_policy_facts(es)  → deadline_days')
print('Tip: hout-06/07 use sar_deadline_days — call get_sar_policy_facts(es) for those decisions.')

In [ ]:
# ── YOUR WORK ── Implement the semantic validator ──────────────────────────────
# Write validate_decision(decision, es) → bool (True if error found).
#
# Steps:
#  1. Retrieve the governing policy rule from cortex-policies using a semantic search.
#     Use the policy_ref field in the decision to identify which policy to check.
#  2. Compare the decision's threshold and direction against the retrieved text.
#  3. Return True if a semantic error is found, False if the decision is correct.
#
# Hints:
# resp = es.search(
#     index='cortex-policies',
#     body={'query': {'semantic': {'field': 'body_semantic',
#                                  'query': f"{decision.get('policy_ref','')} threshold direction"}}},
#     size=3,
# )
# policy_text = ' '.join(h['_source']['body'] for h in resp['hits']['hits'])
# Check: is '10000' or '$10,000' in policy_text? Is 'at or above' in policy_text?
# Compare those to decision['decision']['threshold'] and decision['decision']['direction'].

def validate_decision(decision: dict, es) -> bool:
    """Return True if the decision contains a semantic error."""
    # ── YOUR CODE HERE ──
    return False  # replace with your implementation

In [ ]:
# Run your validator against the dev set
caught = []
false_positives = []
for d in dev_decisions:
    has_error = validate_decision(d, es)
    actual_error = d.get('has_error', False)
    status = ''
    if has_error and actual_error:
        caught.append(d['decision_id']); status = '✓ caught'
    elif has_error and not actual_error:
        false_positives.append(d['decision_id']); status = '✗ false positive'
    elif not has_error and actual_error:
        status = '✗ missed'
    else:
        status = '✓ correct'
    print(f"  {d['decision_id']}: {status}")

print(f"\nDev: caught={len(caught)}, false_positives={len(false_positives)}")
if len(false_positives) == 0 and len(caught) > 0:
    print('Dev set passes. Run the record cell.')
else:
    print('Adjust your validator and re-run.')

In [ ]:
# Evaluate against the graded eval set and record results
# The 10-item eval set is staged by the lab environment (answers not shown).
# Run this after your dev-set pass — it's what the Check scores.
eval_path = pathlib.Path('/home/elastic/dev-sets/semantic-validator-eval-set.jsonl')
eval_decisions = [json.loads(l) for l in eval_path.read_text().splitlines() if l.strip()]

eval_caught = []
for d in eval_decisions:
    if validate_decision(d, es):
        eval_caught.append(d['decision_id'])

print(f"Eval set: flagged {len(eval_caught)} of {len(eval_decisions)} decisions as errors")
print("Tip: hout-06/07 use 'sar_deadline_days'/'filing_required' decision shapes — handle both policy types.")

(TRACES / 'validator-results.json').write_text(json.dumps({
    'caught_errors': eval_caught,
    'false_positives': [],
    'total_checked': len(eval_decisions),
    'timestamp': time.time(),
}, indent=2))
print('Results recorded. Select Check in the sidebar.')
